In [4]:
import pandas as pd
import numpy as np
from src.config import PROCESSED_DATA_DIR, OUTPUT_DATA_DIR

# 1. Load the data
# Adjust 'Date' to whatever your date column is named, or adjust the index logic.
df_obs = pd.read_csv(PROCESSED_DATA_DIR / "combined_streamflow.csv", index_col='Date', parse_dates=True) # m^3/s
df_sim = pd.read_csv(OUTPUT_DATA_DIR / "test_set_predictions.csv", index_col=0, parse_dates=True)  # mm/day
df_attr = pd.read_csv(PROCESSED_DATA_DIR / "static_attributes.csv", index_col=0) # Stations as index

# Find the common stations across all three datasets
common_stations = list(set(df_obs.columns) & set(df_sim.columns) & set(df_attr.index))
print(f"Found {len(common_stations)} common stations.")

# Filter datasets to keep only common stations
df_obs = df_obs[common_stations]
df_sim = df_sim[common_stations]
df_attr = df_attr.loc[common_stations]

# 2. Convert ground truth units from m^3/s to mm/day
# Assumption: Area in attributes is in km^2. 
# 1 m^3/s = (1 * 86400 s/day) / (Area * 1e6 m^2) * 1000 mm/m = 86.4 / Area (km^2)
for station in common_stations:
    area_km2 = df_attr.loc[station, 'basin_area_km2']
    df_obs[station] = df_obs[station] * (86.4 / area_km2)

# 3. Align dates (1980-2022 vs 2013-2022)
# The inner join on indices will automatically restrict both dataframes to 2013-2022
df_obs, df_sim = df_obs.align(df_sim, join='inner')

# 4. Define NSE function
def calculate_nse(obs, sim):
    """Calculates the Nash-Sutcliffe Efficiency."""
    # Drop NaNs to ensure calculation works properly
    valid_mask = ~(np.isnan(obs) | np.isnan(sim))
    obs_valid = obs[valid_mask]
    sim_valid = sim[valid_mask]
    
    if len(obs_valid) == 0:
        return np.nan
        
    numerator = np.sum((obs_valid - sim_valid) ** 2)
    denominator = np.sum((obs_valid - np.mean(obs_valid)) ** 2)
    
    if denominator == 0: # Handle edge case of constant observed flow
        return np.nan
        
    return 1 - (numerator / denominator)

# 5. Calculate NSE for each station
nse_results = {}
for station in common_stations:
    obs_vals = df_obs[station].values
    sim_vals = df_sim[station].values
    nse_results[station] = calculate_nse(obs_vals, sim_vals)

nse_series = pd.Series(nse_results).dropna()

# 6. Report the statistics
print("\n--- NSE Summary Statistics ---")
print(f"Mean:   {nse_series.mean():.3f}")
print(f"Median: {nse_series.median():.3f}")
print(f"Max:    {nse_series.max():.3f}")
print(f"Min:    {nse_series.min():.3f}")
print("\n--- Quartiles ---")
print(f"25th Percentile (Q1): {nse_series.quantile(0.25):.3f}")
print(f"75th Percentile (Q3): {nse_series.quantile(0.75):.3f}")

# Optional: You can save the NSE values for each station to a new CSV
nse_series.to_csv(OUTPUT_DATA_DIR / 'station_nse_results.csv', header=['NSE'])

Found 269 common stations.

--- NSE Summary Statistics ---
Mean:   0.608
Median: 0.728
Max:    0.964
Min:    -1.521

--- Quartiles ---
25th Percentile (Q1): 0.451
75th Percentile (Q3): 0.863


In [3]:
print(nse_series)

08KH001    0.905778
08KE016    0.703932
07EC003    0.881226
05DA010    0.872655
05DB005    0.492668
             ...   
05DF003    0.462558
05DD009    0.517333
08GA061    0.424760
08NL038    0.849188
07EE007    0.929132
Length: 269, dtype: float64
